# M2C2 — NumPy and Pandas for Data Work
**One dataset · one Colab notebook · your own results**

**Today:** Open a table → select and calculate → check data → ask AI for one operation → verify it → save.

## Before you start
1. Open this notebook in **Google Colab** and click **Connect**. Keep your personal copy in Drive.
2. Follow the steps from top to bottom. **C03 loads the classroom Superstore automatically** if `superstore_source.csv` is not in the Colab Files panel. The issued CSV is a backup if the internet download fails. Do not search for a different dataset.
3. At **C12**, stop and use the browser AI to write one small pandas calculation. You will run it yourself.
4. Download your two output CSVs at the end. Colab runtime files can disappear when the session ends.

### How to read this notebook
- **Learn** — short explanation of a command and its syntax.
- **Try** — run or change a small example yourself.
- **Tricky question** — think first, then open the answer to check your reasoning.
- **Prepared step (run, do not edit)** — technical code provided for you; focus on its input and output, not every symbol.

You need **no Kaggle login, paid API, terminal setup or extra starter files**. The AI chat sees only the context you paste, not your running Colab notebook.

**Data note:** One source row is a product line, not necessarily a whole order. The course CSV has `Sales`, `Category`, `Customer ID`, `Order ID`, `Order Date` and `Ship Date`. The dates use `YYYY-MM-DD` (example: `2019-01-03`). Source: [public teaching CSV](https://drive.google.com/file/d/1tyeZkWXo3Lhs5xVk3dDJYpfvBDaUYR5K/view). This is sample teaching data, not verified sales of a real company.

**Do not overwrite `superstore_source.csv`.** Your two output filenames will be `superstore_cleaned.csv` and `customer_summary.csv`.

---
## STEP 1 — Numbers with NumPy (Slide 3)
### C01 — Import libraries
**Why?** A library gives Python extra tools. NumPy works with collections of numbers; pandas works with tables.

**Learn this syntax:** `import numpy as np` means “load NumPy and call it `np`.” Similarly, `import pandas as pd` lets us type `pd` when using pandas. `print(...)` shows a result.

**Try:** Run the cell. No need to memorize library versions.

In [ ]:
# C01 — the two tools for today's lesson
import numpy as np
import pandas as pd
print("Libraries are ready!")

### C02 — Create and change an array
**Learn:** `numbers = np.array([10, 20, 30])` stores three values in `numbers`. The `=` sign **assigns a value to a name**. Brackets `[...]` hold a list; `np.array(...)` makes it a NumPy array.

- `numbers * 2` multiplies **every** value by two.
- `numbers.mean()` asks for the arithmetic average of the original values.
- A dot and `()` like `.mean()` means “run this operation on this object.”

**Try:** Guess both outputs, run the cell, then change `2` to `3` and run again.

In [ ]:
# C02 — your first array
numbers = np.array([10, 20, 30])
print("Original:", numbers)
print("Times two:", numbers * 2)
print("Original average:", numbers.mean())

**Tricky question — think first:** If we multiply the array by 2, does `numbers.mean()` automatically become 40?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. `numbers` still contains 10, 20, 30. `numbers * 2` displays new calculated values without changing the original array. Its original mean is 20.

</details>

---
## STEP 2 — Open and explore the table (Slides 4–5)
### C03 — Load the exact class CSV
**Prepared step — run, do not edit.** It first checks for `superstore_source.csv` in the Colab Files panel. If absent, it downloads the *same classroom version* and checks it. If both methods fail, ask your mentor for the issued CSV and upload it with **Files → Upload**.

**New idea:** A CSV file is a table stored as text. `pd.read_csv(...)` opens it in pandas. Here the loading and checking are already prepared: your job is to run the cell and see that a table appeared.

Do not worry about web requests or file fingerprints inside this setup cell. They are not today's lesson.

In [ ]:
# @title C03 · Load class CSV (run this prepared setup) { display-mode: "form" }
from pathlib import Path
# C03 — local CSV first; otherwise download the SAME classroom file
# Prepared setup: run this cell as it is. No extra package or account is needed.
import hashlib
from io import BytesIO
from http.client import HTTPException
from urllib.request import Request, urlopen

source_path = Path("superstore_source.csv")
expected_sha256 = "3736ad8611053f35d010db944c1c57d5b96cd23a09a0dab68e6d5a06045f1279"
df = None  # Do not keep an old table if this load fails.
source_bytes = None
needs_download = not source_path.exists()

if needs_download:
    print("CSV not found locally. Downloading the class file...")
    # The source project on GitHub hosts this exact CSV on public Google Drive.
    # These are two download routes to ONE file, not different Superstore versions.
    urls = [
        "https://drive.usercontent.google.com/download?id=1tyeZkWXo3Lhs5xVk3dDJYpfvBDaUYR5K&export=download&confirm=t",
        "https://drive.google.com/uc?export=download&id=1tyeZkWXo3Lhs5xVk3dDJYpfvBDaUYR5K&confirm=t",
    ]
    for url in urls:
        try:
            request = Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urlopen(request, timeout=20) as response:
                data = response.read(2_258_217)  # Class file size + 1: reject oversized responses.
            if hashlib.sha256(data).hexdigest() != expected_sha256:
                raise ValueError("The download is not the unchanged class CSV.")
            break
        except (OSError, HTTPException, ValueError) as exc:
            download_error = exc
    else:
        raise RuntimeError(
            "Automatic download failed or the online file changed. Nothing was saved. "
            "In Colab: left Files panel → Upload → choose the issued superstore_source.csv "
            "from the course chat, wait, then rerun C03. Do not use a different Superstore file."
        ) from download_error
else:
    data = source_path.read_bytes()

# Both manual upload and automatic download must match the issued file exactly.
if hashlib.sha256(data).hexdigest() != expected_sha256:
    raise ValueError(
        "This local CSV differs from the issued class file. It was NOT overwritten. "
        "In Files, rename it to keep a backup, then rerun C03 for automatic download "
        "or upload the unchanged superstore_source.csv from the course chat."
    )

loaded_df = pd.read_csv(BytesIO(data), encoding="utf-8", low_memory=False)
required = ["Sales", "Quantity", "Discount", "Profit", "Category", "Region",
            "Customer ID", "Order ID", "Order Date", "Ship Date"]
missing_fields = [name for name in required if name not in loaded_df.columns]
if missing_fields:
    raise ValueError("Wrong CSV. Use the issued file. Missing fields: " + str(missing_fields))
if not pd.api.types.is_numeric_dtype(loaded_df["Sales"]):
    raise ValueError("Sales must be numeric. Ask the mentor to check the file.")

if needs_download:
    # Save only after validation. Exclusive creation never overwrites an existing file.
    with source_path.open("xb") as file:
        file.write(data)
source_bytes = data  # C18 checks that the original bytes stay unchanged.
df = loaded_df
print("Loaded:", source_path.resolve())
print("Source:", "automatic download" if needs_download else "local CSV", "| Class file verified.")

### C04 — Meet your pandas DataFrame
**Learn these commands:**
- `df.head()` — show the **first five rows**. Empty `()` means no additional input is needed.
- `df.shape` — show `(rows, columns)`. **No `()` here:** we are reading information, not running a method.
- `df.columns` — list the actual column names.
- `df.info()` — optional prepared overview of column types (text, numbers, etc.).

**Try:** Run the cell. Point to `Sales`, `Category` and `Order ID`. Remember: the picture on a slide may be only an example; these are the real column names.

In [ ]:
# C04 — see the table before changing it
print("Rows and columns:", df.shape)
print("Column names:", list(df.columns))
display(df.head())
print("Column types:")
df.info()

### C05 — Select, calculate and filter
**Learn the syntax:**
- `df["Sales"]` selects **one column**. A column name is text, so it goes in quotation marks.
- `df[["Category", "Sales"]]` selects **two columns**. The *double brackets* mean a list of names is inside.
- `.sum()` adds the values; `.mean()` gives their average; `len(df)` counts rows.
- `df[df["Category"] == "Technology"]` keeps only rows where the category matches. `==` asks whether values are **equal**; it is not the same as `=`.

**Try:** Run the next two small cells. Then change `"Technology"` to `"Furniture"` and compare the filtered rows.

In [ ]:
# C05A — select columns and calculate
print("Rows:", len(df))
print("Sales total:", df["Sales"].sum())
print("Average Sales per product line:", df["Sales"].mean())
display(df[["Category", "Sales"]].head())

In [ ]:
# C05B — change Technology to Furniture
selected_category = "Technology"
selected_rows = df[df["Category"] == selected_category]
display(selected_rows[["Category", "Sales"]].head())

**Tricky question — think first:** A table has 9,994 rows. Does it necessarily contain 9,994 different orders?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. A row is one product line. A single order can have multiple lines with the same Order ID.

</details>

---
## STEP 3 — Inspect before cleaning (Slides 5–7)
### C06 — Look for missing values and repeated rows
**Learn:** `df.isna()` marks empty values, and `.sum()` counts them for each column. `df.duplicated().sum()` counts fully repeated rows **after the first copy**.

**Important:** Two rows with the same `Order ID` are not automatically duplicate product lines. Today we **look first**; we do not remove anything just to make the table cleaner.

**Try:** Run the checks. Note which missing counts are zero. No cleaning is needed when nothing is missing.

In [ ]:
# C06 — basic data checks, no deletion
print("Empty cells by column:")
print(df.isna().sum())
print("Fully repeated rows:", df.duplicated().sum())

### C07 — Dates, customers and orders
**Learn:** `.nunique()` counts **different values** in a column. Compare it with `len(df)`, which counts product-line rows.

**Important:** If the number of product-line rows is larger than the number of different orders, then some orders have more than one product line.

**Try:** Look at the dates and count the distinct customers and orders.

In [ ]:
# C07 — see dates and count unique IDs
print("Date examples:")
display(df[["Order Date", "Ship Date"]].head())
print("Product-line rows:", len(df))
print("Different customers:", df["Customer ID"].nunique())
print("Different orders:", df["Order ID"].nunique())

### C08 — Your audit note
Write **two real observations** here after running C06–C07:

1. I checked the table and found: **[my actual observation]**.
2. Before changing data, I should: **[my decision and reason]**.

A correct observation can be “there are zero missing values.” Do not invent a problem to fix.

**Tricky question — think first:** Two rows have the same Order ID. Should you delete one row immediately?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. One order may contain more than one product line. The same ID is not proof of an error. Investigate before changing the data.

</details>

---
## STEP 4 — Group rows and ask AI for one thing (Slides 8–9)
### C09 — First, group just three fictional rows
**Learn:** `small.groupby("Category")["Sales"].sum()` means: collect rows with the same `Category`, select their `Sales`, and add them.

The **example numbers are fictional**, not results from our Superstore file.

**Try:** Before you run, predict total Sales for A and B: `(A, 100)`, `(A, 50)`, `(B, 80)`.

In [ ]:
# C09 — tiny example, not the real dataset
small = pd.DataFrame({"Category": ["A", "A", "B"], "Sales": [100, 50, 80]})
display(small)
print(small.groupby("Category")["Sales"].sum())

### C10 — Show the browser AI only the information it needs
**Prepared step — run and copy the printed text.** It shows the real column names, simple data types and three rows of this **public teaching example**. The AI chat cannot automatically open your Colab files.

**Learn:** `df.head(3)` means first **three** rows. Here we print a small, controlled view. Do not paste private data into a public AI chat.

In [ ]:
# C10 — a short and safe context to copy
print("Columns:", list(df.columns))
print("Types for this task:")
print(df[["Category", "Sales"]].dtypes)
print("One row means one product line.")
print("Example rows:")
print(df[["Category", "Sales"]].head(3).to_string(index=False))

### C11 — Write your OWN AI request first
Write one short question for a browser AI. It should mention:
- The table is already named `df` in Colab.
- The real columns are `Category` and `Sales`.
- The task is **add Sales for each Category**.
- Do not overwrite the original data. Explain one way to check the result.

**My first request:** [write your version here]

<details><summary>Optional support — open only after your own attempt</summary>

> In Colab I have a pandas table named `df`. It has the columns `Category` and `Sales`. One row is one product line. Write short pandas code to add up Sales by Category and put the result in `category_totals`. Please keep it as a Series with Category names as its labels. Do not change `df` or overwrite any file. Tell me how I can check one category and the total. I will run the code myself.

</details>

**Check before using AI code:** Is `Sales` spelled correctly? Is the operation `.sum()` rather than `.mean()`? Does it keep the original table unchanged?

### C12 — Your AI-assisted code goes here
**Your turn:** paste the short calculation from the AI response, **after checking its column names**. The result should be stored in `category_totals`.

The placeholder is intentional; your task is to replace it. Do not use Run all until you have completed this cell. If the AI produces complicated code, ask it to simplify to a single `groupby` calculation.

In [ ]:
# C12 — REPLACE the placeholder with your checked AI-assisted code
category_totals = None  # Replace this line with your group-by-Sales calculation.
print(category_totals)

**Tricky question — think first:** The AI produces a convincing table without running your notebook. Is that a verified result?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. You must run the code on your real df and independently check the total. A good explanation is not evidence of a successful calculation.

</details>

---
## STEP 5 — Check the answer yourself (Slide 10)
### C13 — One category, two routes
**Learn:** We can total one category without using `groupby`: filter the original rows, then call `.sum()`. Compare that number with the grouped result.

- `df[df["Category"] == chosen_category]` filters the original rows.
- `filtered["Sales"].sum()` adds their values.
- `category_totals[chosen_category]` reads that category in the grouped result.

**Prepared check:** The last lines allow a tiny computer rounding difference. Focus on *why we compare results*, not on how this technical tolerance works.

**Try:** Read both printed totals. If they disagree, check `Sales` vs `Profit`, `.sum()` vs `.mean()` and the category name.

In [ ]:
# C13 — independent calculation on ORIGINAL rows
if category_totals is None:
    raise ValueError("Complete C12 with your own AI-assisted code first.")
chosen_category = "Technology"
filtered = df[df["Category"] == chosen_category]
from_rows = filtered["Sales"].sum()
from_group = category_totals[chosen_category]
print("Original rows:", from_rows)
print("Grouped result:", from_group)
print("Category check:", np.isclose(from_rows, from_group))
print("All Sales check:", np.isclose(category_totals.sum(), df["Sales"].sum()))

**Tricky question — think first:** The grouped numbers look reasonable, but their grand total is lower than df["Sales"].sum(). Should you accept them?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. Investigate missing categories, filtered rows, or wrong operations. Correct-looking numbers are not enough when totals disagree.

</details>

---
## STEP 6 — Prepare a working copy (Slide 11)
### C14 — Never edit the original
**Learn:** `work = df.copy()` makes a separate working table. We keep `df` for checking the source. The text after `#` in a Python code cell is a **comment**, not a command.

**Try:** Run the cell. Compare the number of rows. Do not remove the repeated row without a clear reason.

In [ ]:
# C14 — keep the source; make a working copy
work = df.copy()
print("Original rows:", len(df))
print("Working rows:", len(work))
# Audit decision: keep all rows unless a correction is justified.

### C15 — Turn text dates into real dates
**Prepared step — run, do not edit.** The source uses dates like `2019-01-03`: year-month-day.

**Learn:** `pd.to_datetime(...)` converts date text to dates; `format="%Y-%m-%d"` tells pandas the order (year-month-day). The command is shown twice because the file has **two date columns**.

**Try:** Compare how the values look before (C07) and after this cell.

In [ ]:
# C15 — two date columns, one confirmed format
work["Order Date"] = pd.to_datetime(work["Order Date"], format="%Y-%m-%d")
work["Ship Date"] = pd.to_datetime(work["Ship Date"], format="%Y-%m-%d")
display(work[["Order Date", "Ship Date"]].head())

### C16 — One row per customer (prepared step)
Our table contains **product lines**. We now want a new table with **customers**.

- **total_spending:** add all Sales for one customer.
- **order_frequency:** count that customer's **different** Order IDs, not product lines.
- **avg_order_value:** total_spending divided by order_frequency.

**Prepared code — run and read the output.** The `agg` syntax is provided; learning its details can wait. Here, explain what each **result column** means, not the full programming syntax.

In [ ]:
# C16 — prepared customer summary, run as it is
customer_summary = work.groupby("Customer ID").agg(
    total_spending=("Sales", "sum"),
    order_frequency=("Order ID", "nunique")
).reset_index()
customer_summary["avg_order_value"] = (
    customer_summary["total_spending"] / customer_summary["order_frequency"]
)
display(customer_summary.head())

**Tricky question — think first:** One customer buys a $20 shirt and $50 trousers in the SAME order. Are their average order value and average product-line Sales both $35?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. There is one $70 order, so average order value is $70. The average of the two product lines is $35, which answers a different question.

</details>

---
## STEP 7 — Save two CSV files (Slide 11)
### C17 — Write CSV outputs
**Learn:** `work.to_csv("name.csv", index=False)` saves a pandas table as a CSV file. The name in quotes is the filename. `index=False` avoids adding an extra row-number column.

Run this cell to create **two new files**, then download them from Colab **Files → ⋮ → Download** after checking them in STEP 8. **Do not save over `superstore_source.csv`.**

In [ ]:
# C17 — two outputs, not a changed source
work.to_csv("superstore_cleaned.csv", index=False, date_format="%Y-%m-%d")
customer_summary.to_csv("customer_summary.csv", index=False)
print("Saved: superstore_cleaned.csv, customer_summary.csv")

**Tricky question — think first:** If your notebook is saved in Google Drive, are the new CSV files automatically saved there too?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. Runtime files may disappear when Colab disconnects. Download both CSV files or explicitly save them to your own Drive folder.

</details>

---
## STEP 8 — Reopen, check and explain (Slides 12–13)
### C18 — Check the saved files
**Learn:** `pd.read_csv("name.csv")` can open a file you just saved. `len(table)` counts its rows, and `.sum()` checks a numeric column total.

**Important:** These two CSVs have **different row meanings**. The cleaned CSV has product lines; the customer CSV has one row per customer. Their row counts should **not** be equal.

**Prepared check:** The code prints whether the original CSV stayed unchanged and totals match. It does not claim a student's work passed unless this cell actually runs.

In [ ]:
# C18 — open the TWO saved outputs
cleaned_again = pd.read_csv("superstore_cleaned.csv")
customers_again = pd.read_csv("customer_summary.csv")
print("Product lines:", len(cleaned_again), "| same as working copy:", len(cleaned_again) == len(work))
print("Customers:", len(customers_again), "| one row per customer:", len(customers_again) == work["Customer ID"].nunique())
print("Sales totals match:", np.isclose(cleaned_again["Sales"].sum(), customers_again["total_spending"].sum()))
print("Source file unchanged:", source_path.read_bytes() == source_bytes)

### C19 — Check one customer (prepared step)
Look at one customer in the saved table. Find the same customer in `work` and compare total spending, different orders and average order value. **Do not memorize the code**: explain why the number of **orders**, not product lines, is used for the average.

In [ ]:
# C19 — prepared independent customer check
customer = customer_summary["Customer ID"].iloc[0]
customer_rows = work[work["Customer ID"] == customer]
spending = customer_rows["Sales"].sum()
orders = customer_rows["Order ID"].nunique()
print("Customer:", customer)
print("Product lines:", len(customer_rows), "| Different orders:", orders)
print("Spending:", spending, "| Average order value:", spending / orders)
saved_row = customers_again[customers_again["Customer ID"] == customer].iloc[0]
print("Customer checks:",
      np.isclose(spending, saved_row["total_spending"]),
      orders == saved_row["order_frequency"],
      np.isclose(spending / orders, saved_row["avg_order_value"]))

### C20 — Explain your own result (no AI answer)
Write 2–3 short sentences after your major notebook steps:
1. What does one row mean in our CSV?
2. Which simple pandas commands did you use yourself?
3. What did your data check show? What did you **not** change, and why?
4. Which Category total did you independently verify?
5. Why is average order value not the average Sales of product lines?
6. Where did you save your two CSV files and this notebook?

**Homework:** Finish the existing LMS **Python Data Preparation Lab** using this same notebook, with real outputs and explanations. Submit the executed Colab link and `superstore_cleaned.csv` as required by LMS before the next class. Keep `customer_summary.csv` for later work. No extra report is required.

**Next class: M2C3 — Data Visualization.** We will use the same checked `superstore_cleaned.csv`.

**Tricky question — think first:** The cleaned CSV has many more rows than the customer summary. Is one of your files necessarily wrong?

<details><summary>Check your reasoning after answering</summary>

**Answer:** No. They represent different things: one row per product line vs one row per customer. Check each file against its own expected count.

</details>